# Cloud Data Fusion — Hands-On Demo (Colab)

This notebook automates everything that **can** be automated for the demo in Section 7 of the
*"Cloud Data Fusion — A Practical Handbook"*: enabling APIs, creating the demo bucket and CSV,
creating a Developer-edition Cloud Data Fusion instance, and granting the IAM roles it needs.

Cloud Data Fusion's Wrangler and Pipeline Studio are **visual, browser-based tools** — there is
no API for "click this column and choose Uppercase," so that part of the demo happens in the
Cloud Data Fusion UI itself, in a regular browser tab. This notebook gets everything ready so
that when you open the UI, you can go straight to wrangling data, and then comes back at the
end to verify the pipeline's output in BigQuery and to clean up.

## Before you start

- You need a GCP project with **billing enabled** and permission to create Cloud Data Fusion,
  Cloud Storage, BigQuery, and IAM resources.
- This notebook creates real, billed resources — most notably the Cloud Data Fusion instance,
  which bills hourly while it exists. Run the **Cleanup** cell at the end when you're done.
- Instance creation alone commonly takes **15–30 minutes**. Kick off Section 4 early and use
  the wait time to read the handbook's Sections 5–6 (plugins and Wrangler) if you haven't yet.
- Matches: handbook Section 7, Steps 1–11.

## 1. Install client libraries and define a small `gcloud` helper

In [ ]:
!pip install --quiet google-cloud-storage google-cloud-bigquery

In [ ]:
import subprocess, json, os, time

def run(cmd, check=True):
    """Run a gcloud/bq/gsutil command (list of args), print it, return stdout."""
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode != 0:
        print(result.stderr.strip())
        if check:
            raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    return result.stdout.strip()

## 2. Authenticate and set your project

`auth.authenticate_user()` authenticates this Colab session both for Google Cloud client
libraries **and** for the `gcloud` CLI calls used throughout this notebook.

In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = ""  #@param {type:"string"}
REGION = "us-central1"  #@param {type:"string"}

assert PROJECT_ID, "Set PROJECT_ID above before continuing."

run(["gcloud", "config", "set", "project", PROJECT_ID])
result = run(["gcloud", "projects", "describe", PROJECT_ID, "--format=value(projectNumber)"])
PROJECT_NUMBER = result
print(f"Using project: {PROJECT_ID} (number {PROJECT_NUMBER})  region: {REGION}")

## 3. Enable the required APIs

Matches handbook Step 1. `dataproc.googleapis.com` is required even though the underlying
service is now branded "Managed Service for Apache Spark" — the API name didn't change
(handbook Section 2's note on the rebrand).

In [ ]:
run(["gcloud", "services", "enable",
     "datafusion.googleapis.com", "storage.googleapis.com",
     "bigquery.googleapis.com", "dataproc.googleapis.com",
     f"--project={PROJECT_ID}"])

In [ ]:
INSTANCE_NAME = "cdf-demo"
BUCKET_NAME = f"{PROJECT_ID}-datafusion-demo"
BQ_DATASET = "datafusion_demo"
BQ_TABLE = "orders_clean"

print("Resource names set.")

## 4. Create a small demo CSV and upload it to Cloud Storage

Matches handbook Step 2. The CSV is deliberately messy — inconsistent casing in `status`, one
negative `amount`, and one missing `status` — so the Wrangler steps in Section 6 below have
something real to fix.

In [ ]:
csv_content = """order_id,customer_name,amount,status
1001,Asha Rao,1499.00,placed
1002,Vikram Iyer,799.50,PLACED
1003,Meera Nair,-50.00,cancelled
1004,Rohan Das,2199.00,shipped
1005,Divya Menon,,placed
"""

with open("orders_raw.csv", "w") as f:
    f.write(csv_content)

print(csv_content)

In [ ]:
run(["gcloud", "storage", "buckets", "create", f"gs://{BUCKET_NAME}",
     f"--location={REGION}", "--uniform-bucket-level-access", f"--project={PROJECT_ID}"])

run(["gcloud", "storage", "cp", "orders_raw.csv",
     f"gs://{BUCKET_NAME}/raw/orders_raw.csv"])

## 5. Create a Developer-edition Cloud Data Fusion instance

Matches handbook Step 3. This is the slow part — instance creation commonly takes
**15–30 minutes**. The cell below kicks off creation and then polls until it's ready, so you
can start it and go read the handbook's Wrangler section (6) while you wait.

In [ ]:
run(["gcloud", "data-fusion", "instances", "create", INSTANCE_NAME,
     f"--location={REGION}", "--edition=developer", "--version=6.11",
     f"--project={PROJECT_ID}", "--async"])

print("Instance creation kicked off asynchronously. Polling for readiness...")

In [ ]:
def get_instance_state():
    return run(["gcloud", "data-fusion", "instances", "describe", INSTANCE_NAME,
                 f"--location={REGION}", f"--project={PROJECT_ID}",
                 "--format=value(state)"], check=False)

for i in range(60):
    state = get_instance_state()
    print(f"[{i*30}s] Instance state: {state}")
    if state == "RUNNING":
        break
    if state in ("FAILED", "DELETING"):
        raise RuntimeError(f"Instance entered unexpected state: {state}")
    time.sleep(30)
else:
    print("Instance did not reach RUNNING in the expected time — check the console.")

In [ ]:
API_ENDPOINT = run(["gcloud", "data-fusion", "instances", "describe", INSTANCE_NAME,
                     f"--location={REGION}", f"--project={PROJECT_ID}",
                     "--format=value(apiEndpoint)"])
print("Cloud Data Fusion API endpoint:", API_ENDPOINT)

console_url = (
    f"https://console.cloud.google.com/data-fusion/instances/details/{REGION}/"
    f"{INSTANCE_NAME}?project={PROJECT_ID}"
)
print("Open this in the console, then click 'View Instance' to reach the Cloud Data Fusion UI:")
print(console_url)

## 6. Grant the Data Fusion service account the permissions it needs

Matches handbook Step 4. Cloud Data Fusion runs pipelines using a Google-managed service agent
that needs permission to launch Spark clusters and access your project's data.

In [ ]:
DATAFUSION_SA = f"service-{PROJECT_NUMBER}@gcp-sa-datafusion.iam.gserviceaccount.com"
COMPUTE_SA = f"{PROJECT_NUMBER}-compute@developer.gserviceaccount.com"

run(["gcloud", "projects", "add-iam-policy-binding", PROJECT_ID,
     f"--member=serviceAccount:{DATAFUSION_SA}",
     "--role=roles/datafusion.serviceAgent", "--condition=None"], check=False)

run(["gcloud", "projects", "add-iam-policy-binding", PROJECT_ID,
     f"--member=serviceAccount:{COMPUTE_SA}",
     "--role=roles/editor", "--condition=None"], check=False)

print("IAM bindings applied. For anything beyond a demo, replace roles/editor on the")
print("compute service account with narrower, purpose-built roles (handbook Section 8.3).")

## 7. Open the Cloud Data Fusion UI and build the pipeline (manual — in your browser)

Matches handbook Steps 5–8. This is the one part of the demo that happens outside this
notebook, because Wrangler and Pipeline Studio are visual tools. The cell above printed the UI
URL — open it in a new browser tab (sign in with the same account you authenticated with in
Section 2), then follow these steps there:

**A. Wrangle the raw CSV**
1. In the left navigation, choose **Wrangler**.
2. **Add Connection → Cloud Storage**, and browse to
   `gs://<PROJECT_ID>-datafusion-demo/raw/orders_raw.csv` (the exact path is printed by the
   cell in Section 4 above).
3. Parse it: column dropdown → **Parse → CSV**, with the first row as a header.
4. Set the `amount` column's type to a decimal/double (column dropdown → **Change data type**).
5. Right-click `status` → **Uppercase**.
6. In the Wrangler command line, add: `filter-rows-on condition-false amount > 0`
7. Add one more: `fill-null-or-empty :status 'UNKNOWN'`
8. Confirm the recipe panel on the right lists all of these directives in order.

**B. Turn the recipe into a pipeline**
1. Click **Create Pipeline** (top right) → **Batch pipeline**.
2. Pipeline Studio opens with a GCSFile source and a Wrangler transform already connected.
3. From the palette, under **Sink**, drag a **BigQuery** stage onto the canvas and connect the
   Wrangler stage's output to it.
4. Configure the BigQuery stage: dataset `datafusion_demo`, table `orders_clean` (Cloud Data
   Fusion creates the dataset if it doesn't exist), write mode at its default.

**C. Deploy and run**
1. Click **Deploy** (validates the schema across every connection).
2. Click **Run**, and watch the status move Provisioning → Running → Succeeded. The first run
   is the slowest, since an ephemeral Spark cluster has to be provisioned from scratch.

Once the run shows **Succeeded**, come back here and continue to Section 8 to verify the
result from this notebook.

## 8. Verify the result in BigQuery

Matches handbook Step 9. Run this after the pipeline in Section 7 has finished successfully.

In [ ]:
from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT_ID)
result_df = bq.query(
    f"SELECT * FROM `{PROJECT_ID}.{BQ_DATASET}.{BQ_TABLE}` ORDER BY order_id"
).to_dataframe()
result_df

You should see four rows: order 1003 (the negative amount) is gone, every `status` value is
upper-case, and order 1005's missing status now reads `UNKNOWN` — exactly what the Wrangler
recipe specified, now running as a repeatable, deployed pipeline.

If the query fails or returns nothing, the pipeline may still be running — check its status in
the Cloud Data Fusion UI before re-running this cell.

## 9. Cleanup

Matches handbook Step 11. Cloud Data Fusion instances are billed hourly while they exist,
whether or not a pipeline is running — delete the instance and supporting resources when
you're done. **This is destructive and cannot be undone** — the checkbox below is a safety
guard.

In [ ]:
CONFIRM_CLEANUP = False  #@param {type:"boolean"}

if CONFIRM_CLEANUP:
    run(["gcloud", "data-fusion", "instances", "delete", INSTANCE_NAME,
         f"--location={REGION}", f"--project={PROJECT_ID}", "--quiet"], check=False)
    run(["bq", "rm", "-r", "-f", "-d", f"{PROJECT_ID}:{BQ_DATASET}"], check=False)
    run(["gcloud", "storage", "rm", "-r", f"gs://{BUCKET_NAME}",
         f"--project={PROJECT_ID}"], check=False)
    print("Cleanup complete.")
else:
    print("Set CONFIRM_CLEANUP = True above and re-run this cell to tear down all resources.")